Paper Pipeline Protoyping

In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
torch.cuda.is_available()

PyTorch version: 2.7.0


True

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_postgres import PGVector
import pandas as pd
import json
import os

In [ ]:
def create_embedding(model, sentences):
  """
  Create embeddings for a list of sentences using the specified model.
  
  Args:
      model: The model to use for creating embeddings.
      sentences: A list of sentences to embed.
  
  Returns:
      A list of embeddings for the input sentences.
  """
  embeddings = model.encode(sentences, convert_to_tensor=True)
  return embeddings

In [29]:
arix_bert_model = SentenceTransformer('ArtifactAI/arxiv-distilroberta-base-GenQ', device='cuda' if torch.cuda.is_available() else 'cpu')
output_embeddings = create_embedding(
  model=arix_bert_model,
  sentences=["This is a test sentence", "This is another test sentence"]
)

In [30]:
print(output_embeddings)

tensor([[-0.2394, -0.6629,  0.5734,  ...,  0.6323, -0.4175, -0.4139],
        [-0.2332, -0.4517,  0.7005,  ...,  0.7400, -0.4372, -0.4405]],
       device='cuda:0')


In [ ]:
def process_paper_metadata_json(model,dataframe):
  """
  Process a JSON file containing paper metadata and extract relevant information.
  
  """
  dataframe["embeddings"] = dataframe.apply(
    lambda row: create_embedding(
      model,
      row['abstract']
    ).tolist(),
    axis=1
  )
  dataframe = dataframe.rename(columns={'abstract': 'context'})
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe[["paper_id", "embeddings", "title", "context"]]
  
  return dataframe


In [41]:
paper_metadata_df = pd.read_json('arxiv-metadata-oai-snapshot.json', lines=True)

In [64]:
paper_metadata_df.head()

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [66]:
dataframe = process_paper_metadata_json(arix_bert_model, paper_metadata_df.head(1))
print(dataframe)

  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. The region of phase space is
specified in which the calculation is most reliable. Good agreement is
demonstrated with data from the Fermilab Tevatron, and predictions are made for
more detailed tests with CDF and DO data. Predictions are shown for
distributions of diphoton pairs produced at the energy of the Large Hadron
Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs
boson are contrasted with those produced from QCD processes at the LHC, showing
that enhanced sensitivity to the signal can be obtained with judicious
selection of events.

    paper_id    

/tmp/ipykernel_34268/1966206868.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe["embeddings"] = dataframe.apply(
